# Random Forest regression to recommend the submodule width range (using multiple features).

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def load_and_prepare_data(file_path):
    """
    Load and preprocess data
    """
    print("📊 Loading data...")
    df = pd.read_excel(file_path)
    print(f"Original data shape: {df.shape}")
    
    # Display basic data information
    print("\nBasic data information:")
    print(df.info())
    
    # Check target variable and key features
    if 'PCE' not in df.columns:
        raise ValueError("Data must contain 'PCE' column")
    
    if 'subcell_width(mm)' not in df.columns:
        raise ValueError("Data must contain 'subcell_width(mm)' column")
    
    # Data cleaning
    df_clean = df.copy()
    
    # Handle missing values
    initial_count = len(df_clean)
    df_clean = df_clean.dropna(subset=['PCE', 'subcell_width(mm)'])
    print(f"After removing missing values: {len(df_clean)} rows (removed {initial_count - len(df_clean)} rows)")
    
    # Filter invalid values
    df_clean = df_clean[df_clean['subcell_width(mm)'] > 0]
    print(f"After filtering invalid widths: {len(df_clean)} rows")
    
    # Select numeric features
    numeric_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    
    # Remove target variable and width features (width feature will be processed separately)
    if 'PCE' in numeric_features:
        numeric_features.remove('PCE')
    if 'subcell_width(mm)' in numeric_features:
        numeric_features.remove('subcell_width(mm)')
    
    print(f"\nAvailable numeric features ({len(numeric_features)}):")
    print(numeric_features)
    
    return df_clean, numeric_features

def prepare_features(df, numeric_features):
    """
    Prepare feature data
    """
    # Build feature matrix
    feature_columns = ['subcell_width(mm)'] + numeric_features
    X = df[feature_columns]
    y = df['PCE']
    
    print(f"\nFeature matrix shape: {X.shape}")
    print(f"Target variable statistics: mean={y.mean():.2f}, std={y.std():.2f}")
    print(f"Sub-cell width range: {X['subcell_width(mm)'].min():.1f}-{X['subcell_width(mm)'].max():.1f} mm")
    
    return X, y, feature_columns

def train_random_forest(X, y, feature_names):
    """
    Train random forest model and evaluate
    """
    print("\n🎯 Training random forest model...")
    
    # Data standardization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split train-test set (use larger test set proportion for small samples)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42
    )
    
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    # Define random forest model (optimized parameters for small samples)
    rf_model = RandomForestRegressor(
        n_estimators=150,           # Increase tree count for stability
        max_depth=6,                # Limit depth to prevent overfitting
        min_samples_split=5,        # Increase minimum split samples
        min_samples_leaf=3,         # Increase minimum leaf node samples
        max_features='sqrt',        # Limit features per tree
        random_state=42,
        n_jobs=-1
    )
    
    # Train model
    rf_model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = rf_model.predict(X_train)
    y_pred_test = rf_model.predict(X_test)
    
    # Evaluate model
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    print("\n📊 Model performance evaluation:")
    print(f"Training set MAE: {train_mae:.3f}, R²: {train_r2:.3f}")
    print(f"Test set MAE: {test_mae:.3f}, R²: {test_r2:.3f}")
    
    # Cross-validation
    if len(X) >= 10:
        cv_scores = cross_val_score(rf_model, X_scaled, y, cv=min(5, len(X)), 
                                  scoring='neg_mean_absolute_error')
        cv_mae = -cv_scores.mean()
        cv_std = cv_scores.std()
        print(f"Cross-validation MAE: {cv_mae:.3f} ± {cv_std:.3f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n🔍 Feature importance ranking:")
    for i, (_, row) in enumerate(feature_importance.iterrows(), 1):
        print(f"  {i:2d}. {row['feature']:20} : {row['importance']:.4f}")
    
    return rf_model, scaler, feature_importance

def predict_optimal_width_range(rf_model, scaler, X, y, feature_names):
    """
    Use trained model to predict optimal width range
    """
    print("\n🔮 Predicting optimal sub-cell width range...")
    
    # Find position of subcell_width in features
    width_idx = feature_names.index('subcell_width(mm)')
    
    # Generate test data: fix other features, vary subcell_width
    width_range = np.linspace(
        X['subcell_width(mm)'].min() * 0.8, 
        X['subcell_width(mm)'].max() * 1.2, 
        200
    )
    
    # Use mean or median of other features (more robust to outliers)
    base_values = X.median().values  # Use median to avoid outlier impact
    
    test_data = []
    for width in width_range:
        test_point = base_values.copy()
        test_point[width_idx] = width
        test_data.append(test_point)
    
    test_data = np.array(test_data)
    
    # Standardize test data
    test_data_scaled = scaler.transform(test_data)
    
    # Predict PCE
    predictions = rf_model.predict(test_data_scaled)
    
    # Find high PCE range (PCE > 18)
    high_pce_threshold = 18.0
    high_pce_mask = predictions >= high_pce_threshold
    high_pce_widths = width_range[high_pce_mask]
    
    if len(high_pce_widths) == 0:
        print("⚠️  No predicted range with PCE>=18% found, using top 80% of predictions")
        # If not reaching 18%, use top 80% of predictions as high PCE range
        pce_threshold = np.percentile(predictions, 80)
        high_pce_mask = predictions >= pce_threshold
        high_pce_widths = width_range[high_pce_mask]
        high_pce_threshold = pce_threshold
    
    # Calculate recommended range
    if len(high_pce_widths) > 0:
        # Theoretical feasible range (all widths with predicted PCE>=18)
        feasible_min = high_pce_widths.min()
        feasible_max = high_pce_widths.max()
        
        # Find point with highest predicted PCE
        peak_idx = np.argmax(predictions)
        peak_width = width_range[peak_idx]
        peak_pce = predictions[peak_idx]
        
        # Calculate dense range: find region where predicted PCE is above 90% of peak
        pce_threshold_90 = peak_pce * 0.90
        dense_mask = predictions >= pce_threshold_90
        dense_widths = width_range[dense_mask]
        
        if len(dense_widths) > 0:
            optimal_min = dense_widths.min()
            optimal_max = dense_widths.max()
        else:
            # If no dense region, use narrow range near peak
            optimal_min = max(feasible_min, peak_width - 1.0)
            optimal_max = min(feasible_max, peak_width + 1.0)
        
        # Further narrow range to conservative recommended range
        conservative_min = max(feasible_min, peak_width - 1.5)
        conservative_max = min(feasible_max, peak_width + 1.5)
        
        result = {
            'width_range': width_range,
            'predictions': predictions,
            'feasible_min': feasible_min,
            'feasible_max': feasible_max,
            'optimal_min': optimal_min,
            'optimal_max': optimal_max,
            'conservative_min': conservative_min,
            'conservative_max': conservative_max,
            'peak_width': peak_width,
            'peak_pce': peak_pce,
            'high_pce_threshold': high_pce_threshold
        }
        
        return result
    else:
        print("❌ Cannot find suitable recommended range")
        return None

def print_recommendation_summary(prediction_result, df):
    """
    Print recommendation result summary
    """
    print("\n" + "="*60)
    print("🎯 Random Forest Recommendation Result Summary")
    print("="*60)
    
    print(f"\n📊 Data overview:")
    print(f"   Total samples: {len(df)}")
    print(f"   Actual PCE range: {df['PCE'].min():.1f} - {df['PCE'].max():.1f}%")
    print(f"   Actual width range: {df['subcell_width(mm)'].min():.1f} - {df['subcell_width(mm)'].max():.1f} mm")
    
    print(f"\n🔮 Model prediction:")
    print(f"   Predicted highest PCE: {prediction_result['peak_pce']:.2f}%")
    print(f"   Predicted optimal width: {prediction_result['peak_width']:.2f} mm")
    
    print(f"\n📏 Recommended range:")
    print(f"   🟡 Theoretical feasible range: {prediction_result['feasible_min']:.1f} - {prediction_result['feasible_max']:.1f} mm")
    print(f"   🟢 Optimal performance range: {prediction_result['optimal_min']:.1f} - {prediction_result['optimal_max']:.1f} mm")
    print(f"   ✅ Conservative recommended range: {prediction_result['conservative_min']:.1f} - {prediction_result['conservative_max']:.1f} mm")
    
    print(f"\n💡 Design suggestions:")
    print(f"   Recommend priority consideration: {prediction_result['conservative_min']:.1f}-{prediction_result['conservative_max']:.1f} mm")
    print(f"   Focus on configurations near: {prediction_result['peak_width']:.1f} mm")
    
    # Check performance of actual data in recommended range
    in_conservative = df[
        (df['subcell_width(mm)'] >= prediction_result['conservative_min']) & 
        (df['subcell_width(mm)'] <= prediction_result['conservative_max'])
    ]
    
    if len(in_conservative) > 0:
        avg_pce = in_conservative['PCE'].mean()
        print(f"\n📈 Validation information:")
        print(f"   Actual data points in conservative range: {len(in_conservative)}")
        print(f"   Average actual PCE in range: {avg_pce:.2f}%")
        
        if avg_pce >= 18:
            print(f"   ✅ Range validation: Actual performance good")
        else:
            print(f"   ⚠️  Range validation: Actual performance average, suggest combining engineering experience")

def main():
    """
    Main function
    """
    try:
        # 1. Load data
        df, numeric_features = load_and_prepare_data('subcell.xlsx')
        
        if len(df) < 10:
            print("⚠️  Small data volume, results are for reference only")
        
        # 2. Prepare features
        X, y, feature_names = prepare_features(df, numeric_features)
        
        # 3. Train random forest model
        rf_model, scaler, feature_importance = train_random_forest(X, y, feature_names)
        
        # 4. Predict optimal width range
        prediction_result = predict_optimal_width_range(rf_model, scaler, X, y, feature_names)
        
        if prediction_result is not None:
            # 5. Print recommendation summary
            print_recommendation_summary(prediction_result, df)
            
            print(f"\n✅ Analysis completed!")
        else:
            print("❌ Analysis failed, cannot generate recommended range")
            
    except Exception as e:
        print(f"❌ Program execution error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    print("=== Sub-cell Width Range Recommendation Based on Random Forest ===")
    main()

=== Sub-cell Width Range Recommendation Based on Random Forest ===
📊 Loading data...
Original data shape: (34, 31)

Basic data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Active_Area                    34 non-null     float64
 1   P1Wavelength(nm)               34 non-null     int64  
 2   P2Wavelength(nm)               34 non-null     int64  
 3   P3Wavelength(nm)               34 non-null     int64  
 4   total_scribing_line_width(μm)  34 non-null     float64
 5   P1Width(μm)                    34 non-null     float64
 6   P2Width(μm)                    34 non-null     float64
 7   P3Width(μm)                    34 non-null     float64
 8   GFF                            34 non-null     float64
 9   Type                           34 non-null     int64  
 10  submodule_number               3

# Statistical analysis (using only 2 features, based on the distribution of high-PCE data points, performed via direct statistics)

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

def analyze_subcell_pce_relationship():
    """
    Analyze the relationship between subcell width and PCE, optimized for small sample data
    """
    try:
        # 1. Read Excel file
        print("📊 Reading subcell.xlsx file...")
        df = pd.read_excel('subcell.xlsx')
        print(f"✅ Successfully read file, total {len(df)} rows of data")
        
        # 2. Check if required columns exist
        required_columns = ['subcell_width(mm)', 'PCE']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            print(f"❌ File is missing required columns: {missing_columns}")
            print(f"   Columns in file: {list(df.columns)}")
            return
        
        # 3. Data preprocessing
        print("🔍 Data preprocessing...")
        # Remove missing values
        df_clean = df.dropna(subset=required_columns)
        print(f"   Data after cleaning missing values: {len(df_clean)} rows")
        
        # Ensure correct data types
        df_clean['subcell_width(mm)'] = pd.to_numeric(df_clean['subcell_width(mm)'], errors='coerce')
        df_clean['PCE'] = pd.to_numeric(df_clean['PCE'], errors='coerce')
        
        # Remove rows with failed conversion
        df_clean = df_clean.dropna(subset=required_columns)
        print(f"   Data after numeric conversion: {len(df_clean)} rows")
        
        # Filter out data points with subcell_width(mm)=0
        df_clean = df_clean[df_clean['subcell_width(mm)'] > 0]
        print(f"   Data after filtering width=0 points: {len(df_clean)} rows")
        
        if len(df_clean) < 5:
            print("❌ Too few valid data points for reliable analysis")
            return
        
        # 4. Sort by subcell width
        df_sorted = df_clean.sort_values('subcell_width(mm)')
        
        # 5. Separate data points with PCE > 18 and <=18
        high_pce_mask = df_sorted['PCE'] > 18
        df_high_pce = df_sorted[high_pce_mask]
        df_low_pce = df_sorted[~high_pce_mask]
        
        print(f"\n📈 Data statistics:")
        print(f"   Total valid data points: {len(df_clean)}")
        print(f"   Data points with PCE > 18%: {len(df_high_pce)}")
        print(f"   Data points with PCE ≤ 18%: {len(df_low_pce)}")
        print(f"   Subcell width range: {df_sorted['subcell_width(mm)'].min():.2f} - {df_sorted['subcell_width(mm)'].max():.2f} mm")
        print(f"   PCE range: {df_sorted['PCE'].min():.2f} - {df_sorted['PCE'].max():.2f} %")
        
        # 11. Detailed analysis and recommendations
        if len(df_high_pce) > 0:
            high_pce_widths = df_high_pce['subcell_width(mm)']
            high_pce_pce = df_high_pce['PCE']
            
            # Calculate recommended interval for small sample data
            if len(df_high_pce) > 0:
                # Calculate statistics for high PCE data
                if len(df_high_pce) >= 3:
                    # Method 1: Using quartiles and mode concept
                    width_median = high_pce_widths.median()
                    q1 = high_pce_widths.quantile(0.25)
                    q3 = high_pce_widths.quantile(0.75)
                    
                    # Calculate data point density (simple method)
                    width_bins = np.linspace(high_pce_widths.min(), high_pce_widths.max(), min(10, len(high_pce_widths)))
                    hist, bin_edges = np.histogram(high_pce_widths, bins=width_bins)
                    max_bin_idx = np.argmax(hist)
                    density_center = (bin_edges[max_bin_idx] + bin_edges[max_bin_idx+1]) / 2
                    
                    # Combine median and density center
                    if abs(width_median - density_center) <= 2:
                        center_point = (width_median + density_center) / 2
                    else:
                        center_point = width_median
                    
                    # Determine interval width based on data volume
                    if len(df_high_pce) >= 5:
                        range_width = max(1.5, (q3 - q1) * 0.8)
                    else:
                        range_width = 1.0  # Use narrower interval for small samples
                    
                    optimal_min = max(high_pce_widths.min(), center_point - range_width)
                    optimal_max = min(high_pce_widths.max(), center_point + range_width)
                    
                else:
                    # Too few data points, use simple method
                    width_median = high_pce_widths.median()
                    optimal_min = max(high_pce_widths.min(), width_median - 1.0)
                    optimal_max = min(high_pce_widths.max(), width_median + 1.0)
                
                # Ensure interval is reasonable
                optimal_min = max(optimal_min, df_sorted['subcell_width(mm)'].min())
                optimal_max = min(optimal_max, df_sorted['subcell_width(mm)'].max())
            
            print(f"\n🎯 Analysis based on {len(df_clean)} data points:")
            print(f"   Number of high PCE data points: {len(df_high_pce)}")
            print(f"   High PCE width range: {high_pce_widths.min():.2f} - {high_pce_widths.max():.2f} mm")
            
            if len(df_high_pce) >= 3:
                width_median = high_pce_widths.median()
                print(f"   High PCE width median: {width_median:.2f} mm")
            
            print(f"   📍 Recommended interval: {optimal_min:.1f} - {optimal_max:.1f} mm")
            
            # Calculate statistics within recommended interval
            in_optimal_zone = df_high_pce[
                (df_high_pce['subcell_width(mm)'] >= optimal_min) & 
                (df_high_pce['subcell_width(mm)'] <= optimal_max)
            ]
            
            if len(in_optimal_zone) > 0:
                optimal_pce_mean = in_optimal_zone['PCE'].mean()
                optimal_pce_std = in_optimal_zone['PCE'].std()
                coverage = len(in_optimal_zone) / len(df_high_pce) * 100
                
                print(f"   High PCE points in recommended interval: {len(in_optimal_zone)} ({coverage:.1f}%)")
                print(f"   Average PCE in recommended interval: {optimal_pce_mean:.2f} ± {optimal_pce_std:.2f} %")
            
            # Correlation analysis
            correlation = df_sorted['subcell_width(mm)'].corr(df_sorted['PCE'])
            print(f"\n📊 Statistical analysis:")
            print(f"   Correlation between subcell width and PCE: {correlation:.3f}")
            
            # Calculate correlation significance (for small samples)
            if len(df_clean) >= 5:
                p_value = stats.pearsonr(df_sorted['subcell_width(mm)'], df_sorted['PCE'])[1]
                significance = "Significant" if p_value < 0.05 else "Not significant"
                print(f"   Correlation significance (p-value): {p_value:.3f} ({significance})")
            
            if correlation > 0.3:
                trend = "Strong positive correlation (width increase clearly beneficial for PCE)"
            elif correlation > 0.1:
                trend = "Weak positive correlation (width increase may be beneficial for PCE)"
            elif correlation < -0.3:
                trend = "Strong negative correlation (width decrease clearly beneficial for PCE)"
            elif correlation < -0.1:
                trend = "Weak negative correlation (width decrease may be beneficial for PCE)"
            else:
                trend = "No clear linear relationship"
            print(f"   Trend: {trend}")
                
            print(f"\n💡 Design suggestions (based on {len(df_clean)} samples):")
            print(f"   Recommend designing subcell width within {optimal_min:.1f}-{optimal_max:.1f} mm range")
            
            if len(df_high_pce) >= 3:
                print(f"   Focus on configurations near {width_median:.1f} mm")
            
            # Notes for small sample data
            if len(df_high_pce) < 5:
                print(f"\n⚠️  Note: Few high PCE data points ({len(df_high_pce)}), suggestions:")
                print(f"   - Collect more data to improve recommendation accuracy")
                print(f"   - Verify recommended interval in actual design")
        
        else:
            print(f"\n⚠️  No data points with PCE>18% found, cannot provide recommended interval")
            print(f"   Suggest lowering PCE threshold or collecting more high PCE data")
        
    except FileNotFoundError:
        print("❌ File 'subcell.xlsx' not found, please ensure file exists in current directory")
    except Exception as e:
        print(f"❌ Error processing file: {e}")

def create_sample_data():
    """
    Create sample data (if actual file does not exist)
    """
    try:
        # Create example data with 34 data points
        np.random.seed(42)  # Ensure reproducibility
        
        # Generate 34 width values, concentrated in certain area
        widths = np.concatenate([
            np.random.uniform(5, 10, 8),
            np.random.uniform(10, 15, 12),
            np.random.uniform(15, 20, 8),
            np.random.uniform(20, 25, 6)
        ])
        
        # Generate corresponding PCE values, with higher PCE in specific width range
        pce_values = []
        for w in widths:
            if 10 <= w <= 15:
                # Higher PCE in 10-15mm range
                pce = 18 + np.random.uniform(0, 1.5)
            else:
                # Lower PCE in other ranges
                pce = 16 + np.random.uniform(0, 2)
            pce_values.append(pce)
        
        sample_data = {
            'subcell_width(mm)': widths,
            'PCE': pce_values
        }
        
        df_sample = pd.DataFrame(sample_data)
        df_sample.to_excel('subcell.xlsx', index=False)
        print("📝 Created sample data file 'subcell.xlsx' with 34 data points")
        print("   Please replace this file with actual data for accurate analysis")
        
    except Exception as e:
        print(f"❌ Failed to create sample data: {e}")

if __name__ == "__main__":
    print("=== Subcell Width and PCE Relationship Analysis (Optimized for Small Samples) ===\n")
    
    # Attempt to analyze data
    analyze_subcell_pce_relationship()
    
    # If file does not exist, create sample data
    try:
        pd.read_excel('subcell.xlsx')
    except:
        print("\n⚠️  No valid data file found, creating sample data with 34 data points...")
        create_sample_data()
        print("\n🔄 Re-attempting analysis...")
        analyze_subcell_pce_relationship()

=== Subcell Width and PCE Relationship Analysis (Optimized for Small Samples) ===

📊 Reading subcell.xlsx file...
✅ Successfully read file, total 34 rows of data
🔍 Data preprocessing...
   Data after cleaning missing values: 34 rows
   Data after numeric conversion: 34 rows
   Data after filtering width=0 points: 34 rows

📈 Data statistics:
   Total valid data points: 34
   Data points with PCE > 18%: 12
   Data points with PCE ≤ 18%: 22
   Subcell width range: 4.00 - 7.00 mm
   PCE range: 3.60 - 24.14 %

🎯 Analysis based on 34 data points:
   Number of high PCE data points: 12
   High PCE width range: 5.00 - 6.80 mm
   High PCE width median: 5.00 mm
   📍 Recommended interval: 5.0 - 6.5 mm
   High PCE points in recommended interval: 10 (83.3%)
   Average PCE in recommended interval: 21.40 ± 2.08 %

📊 Statistical analysis:
   Correlation between subcell width and PCE: 0.112
   Correlation significance (p-value): 0.527 (Not significant)
   Trend: Weak positive correlation (width increase